# 第4章 演習（正解）: Human-in-the-Loop (HITL) の実装

---

In [ ]:
%pip install -qU langchain langchain-openai langgraph

In [ ]:
import os
from google.colab import userdata

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = "chap04-solution"
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
print("環境変数の設定が完了しました")

In [ ]:
from langchain.tools import tool

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """指定したアドレスにメールを送信します。この操作は外部システムに影響を与えます。
    
    Args:
        to: 送信先のメールアドレス
        subject: メールの件名
        body: メールの本文
    """
    print(f"[メール送信] 宛先: {to}, 件名: {subject}")
    return f"メールを送信しました: {to} 宛, 件名: '{subject}'"


@tool
def delete_record(record_id: str, table: str) -> str:
    """指定したテーブルからレコードを削除します。この操作は取り消せません。
    
    Args:
        record_id: 削除するレコードの ID
        table: 対象のテーブル名
    """
    print(f"[データ削除] テーブル: {table}, レコードID: {record_id}")
    return f"レコード {record_id} を {table} テーブルから削除しました"


@tool
def get_user_info(user_id: str) -> str:
    """ユーザー情報を取得します（読み取り専用）。
    
    Args:
        user_id: 情報を取得するユーザーの ID
    """
    return f"ユーザー情報: ID={user_id}, 名前=山田太郎, メール=yamada@example.com"

print("ツールの準備が完了しました")

## 課題1の正解: HITL エージェントを作成する

In [ ]:
# ✅ 正解
from langchain.agents import create_agent
from langchain.middleware import HumanInTheLoopMiddleware   # ← ここでインポート
from langgraph.checkpoint.memory import InMemorySaver

# interrupt_on に承認が必要なツール名をセットで指定
hitl_middleware = HumanInTheLoopMiddleware(
    interrupt_on={"send_email", "delete_record"},
)

hitl_agent = create_agent(
    model="openai:gpt-4o",
    tools=[send_email, delete_record, get_user_info],
    middleware=[hitl_middleware],   # ← middleware として HITL を追加
    checkpointer=InMemorySaver(),  # ← 中断状態保存のため必須
    system_prompt=(
        "あなたはビジネスオペレーション AIエージェントです。\n"
        "メール送信やデータ削除などの重要な操作は、人間の承認を得てから実行してください。\n"
        "情報取得は承認なしで実行できます。\n"
        "日本語で回答してください。"
    ),
)

print("HITL エージェントの作成が完了しました")

## 課題2の正解: Interrupt の発生を確認する

In [ ]:
# ✅ 正解
config = {"configurable": {"thread_id": "hitl_solution_001"}}

result = hitl_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "ユーザーID U999 の情報を取得し、そのメールアドレスに件名『テスト』でメールを送ってください"
            }
        ]
    },
    config=config,
    version="v2",   # interrupt 情報を取得するために必要
)

if result.interrupts:
    print("⚠️ 承認待ちです")
    for interrupt in result.interrupts:
        print(f"  承認待ちのツール: {interrupt['tool_call']['name']}")
        print(f"  引数: {interrupt['tool_call']['args']}")

## 課題3の正解: Approve または Reject して再開する

In [ ]:
# ✅ 正解
from langgraph.types import Command   # ← Command をインポート

user_decision = input("処理を承認しますか？ (approve/reject): ").strip().lower()

if user_decision == "approve":
    print("承認します...")
    # Command(resume=...) で承認してエージェントを再開
    resume_result = hitl_agent.invoke(
        Command(resume={"decisions": [{"type": "approve"}]}),
        config=config,   # 同じ thread_id で再開
        version="v2",
    )
    print(resume_result.messages[-1].content)

elif user_decision == "reject":
    print("拒否します...")
    # Command(resume=...) で拒否してエージェントに通知
    reject_result = hitl_agent.invoke(
        Command(resume={"decisions": [{"type": "reject", "reason": "ユーザーがキャンセルしました"}]}),
        config=config,
        version="v2",
    )
    print(reject_result.messages[-1].content)

else:
    print("approve または reject を入力してください")